# Data Preprocessing

This notebook focuses strictly on the preprocessing logic for the Netflix dataset. It includes handling missing values, transforming multi-value columns, mapping values, and saving the intermediate and final datasets into appropriate directories.

In [3]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

## Setup Directories & Load Data
Ensure the correct folder structures exist and load the original immutable data.

In [4]:
# Set up output directories
os.makedirs('../data/interim', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Load raw data
raw_data_path = '../data/raw/netflix_movies_detailed_up_to_2025.csv'
df = pd.read_csv(raw_data_path)

# Show initial shape
print(f"Initial Dataset Shape: {df.shape}")

Initial Dataset Shape: (16000, 18)


## Basic Cleaning
Drop completely unuseful columns indicating full missing values.
Remove any dataset rows that contain NA references on specific integral columns for consistent modeling later use.

In [5]:
# Drop columns
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])

# Remove missing values on critical identifying columns
columns_to_care = ['country', 'cast', 'director', 'description', 'genres']
df = df.dropna(subset=columns_to_care)

# Optional index reset
df = df.reset_index(drop=True)

# Print out is there any missing values after drops
print("Missing values after drops:")
print(df.isnull().sum())

# Validate missing drops and columns drops
print(f"Data shape after initial drops: {df.shape}")

# Save intermediate dataset
df.to_csv('../data/interim/movies_cleaned.csv', index=False)
print("Saved interim cleaned dataset -> interim/movies_cleaned.csv")

Missing values after drops:
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
genres          0
language        0
description     0
popularity      0
vote_count      0
vote_average    0
budget          0
revenue         0
dtype: int64
Data shape after initial drops: (15239, 17)
Saved interim cleaned dataset -> interim/movies_cleaned.csv


## Language Mapping
Map standard codes into full English names. Unseen entries can just default to 'Unknown'. We retain both columns.

In [6]:
language_mapping = {
    'en': 'English',
    'es': 'Spanish',
    'sr': 'Serbian',
    'pt': 'Portuguese',
    'ko': 'Korean',
    'fr': 'French',
    'hi': 'Hindi',
    'ja': 'Japanese',
    'th': 'Thai',
    'no': 'Norwegian',
    'de': 'German',
    'zh': 'Chinese',
    'cn': 'Chinese',  # non-standard, but keep if exists in data
    'it': 'Italian',
    'fi': 'Finnish',
    'sv': 'Swedish',
    'is': 'Icelandic',
    'ta': 'Tamil',
    'nl': 'Dutch',
    'da': 'Danish',
    'ru': 'Russian',
    'tr': 'Turkish',
    'el': 'Greek',
    'te': 'Telugu',
    'bn': 'Bengali',
    'ar': 'Arabic',
    'ca': 'Catalan',
    'fa': 'Persian',
    'ro': 'Romanian',
    'he': 'Hebrew',
    'cs': 'Czech',
    'hu': 'Hungarian',
    'pl': 'Polish',
    'ml': 'Malayalam',
    'ms': 'Malay',
    'id': 'Indonesian',
    'ku': 'Kurdish',
    'xx': 'Unknown',  # fallback / undefined
    'tl': 'Tagalog',  # FIXED (not Thai)
    'gl': 'Galician',
    'lt': 'Lithuanian',
    'si': 'Sinhala',
    'et': 'Estonian',
    'hr': 'Croatian',
    'ps': 'Pashto',
    'mi': 'Maori',
    'uk': 'Ukrainian',
    'af': 'Afrikaans',
    'km': 'Khmer',
    'mr': 'Marathi',
    'sk': 'Slovak',
    'kn': 'Kannada',
    'eu': 'Basque',
    'lv': 'Latvian',
    'la': 'Latin',
    'dz': 'Dzongkha',
    'kk': 'Kazakh',
    'mk': 'Macedonian',
    'ka': 'Georgian',
    'vi': 'Vietnamese',
    'mn': 'Mongolian',
    'pa': 'Punjabi',
    'ga': 'Irish',
    'yo': 'Yoruba',
    'zu': 'Zulu',
    'ky': 'Kyrgyz',
    'ur': 'Urdu',
    'kl': 'Greenlandic',
    'ht': 'Haitian Creole',
    'am': 'Amharic',
    'ne': 'Nepali',
    'hy': 'Armenian',
    'sl': 'Slovenian',
    'bg': 'Bulgarian'
}

# Map language codes
if 'language' in df.columns:
    df['language_code'] = df['language']
    
    df['language_name'] = df['language_code'].apply(
        lambda x: language_mapping.get(str(x).lower().strip(), 'Unknown') if pd.notna(x) else 'Unknown'
    )
    
    df = df.drop(columns=['language'])
    
# Inspect changes
df[['language_code', 'language_name']].head(10)

# Print all language after mapping
print("Unique Languages in the Dataset after mapping:")
print(df['language_name'].unique())

Unique Languages in the Dataset after mapping:
['English' 'Spanish' 'Serbian' 'Portuguese' 'Korean' 'French' 'Hindi'
 'Japanese' 'Thai' 'Norwegian' 'German' 'Chinese' 'Italian' 'Finnish'
 'Swedish' 'Icelandic' 'Tamil' 'Dutch' 'Danish' 'Russian' 'Turkish'
 'Greek' 'Telugu' 'Bengali' 'Arabic' 'Catalan' 'Persian' 'Romanian'
 'Hebrew' 'Czech' 'Hungarian' 'Polish' 'Malayalam' 'Malay' 'Indonesian'
 'Kurdish' 'Unknown' 'Tagalog' 'Galician' 'Lithuanian' 'Sinhala'
 'Estonian' 'Croatian' 'Maori' 'Ukrainian' 'Afrikaans' 'Khmer' 'Marathi'
 'Slovak' 'Kannada' 'Basque' 'Latvian' 'Latin' 'Dzongkha' 'Kazakh'
 'Macedonian' 'Georgian' 'Vietnamese' 'Mongolian' 'Punjabi' 'Irish'
 'Yoruba' 'Zulu' 'Urdu' 'Amharic' 'Nepali' 'Kyrgyz' 'Slovenian']


## Normalize Multi-Value Columns
We parse `genres`, `country`, and `director` directly into lists.
After parsing, mapping tables bridging tables generated locally will be used.

In [7]:
# Function to split and explode comma-separated columns
def process_bridge_table(data_df, id_col, value_col, new_col_name):
    # Slice the required columns
    temp_df = data_df[[id_col, value_col]].copy()
    
    # Split text by commas 
    temp_df[value_col] = temp_df[value_col].apply(lambda text: [x.strip() for x in str(text).split(',') if x.strip()])
    
    # Convert list into new rows
    exploded_df = temp_df.explode(value_col)
    
    # Clean possible empty slots
    exploded_df = exploded_df.dropna(subset=[value_col])
    exploded_df = exploded_df[exploded_df[value_col] != ""]
    
    # Standardize textual naming cases 
    exploded_df[value_col] = exploded_df[value_col].str.strip()
    
    # Rename targeted output column
    exploded_df = exploded_df.rename(columns={value_col: new_col_name})
    
    return exploded_df

# Handle generation if `show_id` available
if 'show_id' in df.columns:
    
    movie_genres = process_bridge_table(df, 'show_id', 'genres', 'genre')
    movie_countries = process_bridge_table(df, 'show_id', 'country', 'country_name')
    movie_directors = process_bridge_table(df, 'show_id', 'director', 'director_name')
    movie_casts = process_bridge_table(df, 'show_id', 'cast', 'cast_name')

    # Convert features to explicit standard list array values in dataframe directly for downstream reading parsing
    df['genres'] = df['genres'].apply(lambda x: [str(i).strip() for i in str(x).split(',') if i.strip()])
    df['country'] = df['country'].apply(lambda x: [str(i).strip() for i in str(x).split(',') if i.strip()])
    df['director'] = df['director'].apply(lambda x: [str(i).strip() for i in str(x).split(',') if i.strip()])
    df['cast'] = df['cast'].apply(lambda x: [str(i).strip() for i in str(x).split(',') if i.strip()])

    # Save the split multi-value structures
    movie_genres.to_csv('../data/processed/movie_genres.csv', index=False)
    print("Saved bridge table -> processed/movie_genres.csv")
    
    movie_countries.to_csv('../data/processed/movie_countries.csv', index=False)
    print("Saved bridge table -> processed/movie_countries.csv")
    
    movie_directors.to_csv('../data/processed/movie_directors.csv', index=False)
    print("Saved bridge table -> processed/movie_directors.csv")

    movie_casts.to_csv('../data/processed/movie_casts.csv', index=False)
    print("Saved bridge table -> processed/movie_casts.csv")

Saved bridge table -> processed/movie_genres.csv
Saved bridge table -> processed/movie_countries.csv
Saved bridge table -> processed/movie_directors.csv
Saved bridge table -> processed/movie_casts.csv


## Final Delivery
Save the processed payload into the final directory.

In [6]:
# Final preview of structured dataset attributes
print(f"Final Data shape for Output Generation: {df.shape}")
df.head(2)

# Save processed data payload
df.to_csv('../data/processed/movies.csv', index=False)
print("Saved final processed dataset -> processed/movies.csv")

Final Data shape for Output Generation: (15239, 18)
Saved final processed dataset -> processed/movies.csv
